# Resultados consolidados: calidad y representatividad

Este notebook consolida la segunda etapa de EVA1. Lee el dataset original sin modificarlo, genera una versión derivada deduplicada exclusivamente para análisis y exporta las tablas utilizadas como evidencia. No entrena modelos.

In [1]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / 'data' / 'raw' / 'heart.csv').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import pandas as pd
from src.validation_stage2 import export_stage2_outputs
from src.plots import save_original_unique_comparison, save_subgroup_heatmap

RAW_PATH = PROJECT_ROOT / 'data' / 'raw' / 'heart.csv'
PROCESSED_DIR = PROJECT_ROOT / 'data' / 'processed'
TABLES_DIR = PROJECT_ROOT / 'outputs' / 'tables'
FIGURES_DIR = PROJECT_ROOT / 'outputs' / 'figures'
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

outputs = export_stage2_outputs(RAW_PATH, PROCESSED_DIR, TABLES_DIR)
df_original = pd.read_csv(RAW_PATH)
df_unique = pd.read_csv(PROCESSED_DIR / 'heart_unique.csv')
print(f'Original: {len(df_original):,} registros | Únicos: {len(df_unique):,} registros')

Original: 1,025 registros | Únicos: 302 registros


## 1. Caracterización breve y duplicados

Los duplicados se identifican mediante igualdad exacta en las 14 columnas. La tabla conserva explícitamente las categorías de una, dos, tres y más de tres repeticiones; una frecuencia cero también es evidencia útil.

In [2]:
duplicate_table = outputs['analisis_duplicados']
display(duplicate_table)
print(f"Filas duplicadas: {df_original.duplicated().sum():,} ({df_original.duplicated().mean() * 100:.2f}%).")
print(f"Registros únicos: {len(df_unique):,}.")

,categoria_repeticion,frecuencias_incluidas,observaciones_distintas,filas_originales_asociadas,porcentaje_del_dataset_original,maximo_repeticiones_dataset
0,1,1,0,0,0.00,8
1,2,2,0,0,0.00,8
2,3,3,187,561,54.73,8
3,>3,"4, 8",115,464,45.27,8


Filas duplicadas: 723 (70.54%).
Registros únicos: 302.


## 2. Comparación original vs. deduplicado

Las diferencias se expresan en puntos porcentuales (deduplicado menos original). No se concluye que los duplicados sean inocuos sin esta cuantificación.

In [3]:
comparison = outputs['comparacion_original_vs_unicos']
display(comparison)
for dimension, title, xlabel, filename in [
    ('sex', 'Distribución por sex: original vs. deduplicado', 'sex (código)', 'comparacion_original_vs_unicos_sex.png'),
    ('age_group', 'Rangos etarios: original vs. deduplicado', 'Rango etario', 'comparacion_original_vs_unicos_edad.png'),
]:
    plot_data = comparison.loc[comparison['dimension'] == dimension, ['categoria', 'porcentaje_original', 'porcentaje_unicos']].melt(
        id_vars='categoria', var_name='version', value_name='porcentaje'
    ).replace({'porcentaje_original': 'Original', 'porcentaje_unicos': 'Deduplicado'})
    save_original_unique_comparison(plot_data, title, xlabel, FIGURES_DIR / filename)

,dimension,categoria,n_original,porcentaje_original,n_unicos,porcentaje_unicos,diferencia_puntos_porcentuales
0,target,0,499,48.68,138,45.70,-2.99
1,target,1,526,51.32,164,54.30,2.99
2,sex,0,312,30.44,96,31.79,1.35
3,sex,1,713,69.56,206,68.21,-1.35
4,age_group,<40,57,5.56,15,4.97,-0.59
5,age_group,40-49,237,23.12,72,23.84,0.72
6,age_group,50-59,422,41.17,125,41.39,0.22
7,age_group,60-69,275,26.83,80,26.49,-0.34
8,age_group,70+,34,3.32,10,3.31,-0.01


## 3. Representación por sex y edad

Los códigos se conservan como `sex=0` y `sex=1`: no se les asigna significado demográfico porque no hay documentación local que lo establezca. Los porcentajes de target se calculan dentro de cada grupo y se acompañan de N.

In [4]:
display(outputs['representacion_sex'])
display(outputs['representacion_edad'])
display(outputs['target_por_grupo_original_vs_unicos'])

,grupo,categoria,n_original,target_0_n_original,target_0_pct_original,target_1_n_original,target_1_pct_original,n_unicos,target_0_n_unicos,target_0_pct_unicos,target_1_n_unicos,target_1_pct_unicos
0,sex,0,312,86,27.56,226,72.44,96,24,25.00,72,75.00
1,sex,1,713,413,57.92,300,42.08,206,114,55.34,92,44.66


,dataset_evaluado,tipo,categoria,valor,n,porcentaje,target_0_n,target_1_n,target_0_pct_grupo,target_1_pct_grupo
0,original,estadistico,minimo,29.00,NaN,NaN,NaN,NaN,NaN,NaN
1,original,estadistico,maximo,77.00,NaN,NaN,NaN,NaN,NaN,NaN
2,original,estadistico,media,54.43,NaN,NaN,NaN,NaN,NaN,NaN
3,original,estadistico,mediana,56.00,NaN,NaN,NaN,NaN,NaN,NaN
4,original,estadistico,desviacion_estandar,9.07,NaN,NaN,NaN,NaN,NaN,NaN
5,original,rango_etario,<40,NaN,57.0,5.56,15.0,42.0,26.32,73.68
6,original,rango_etario,40-49,NaN,237.0,23.12,80.0,157.0,33.76,66.24
7,original,rango_etario,50-59,NaN,422.0,41.17,216.0,206.0,51.18,48.82
8,original,rango_etario,60-69,NaN,275.0,26.83,174.0,101.0,63.27,36.73
9,original,rango_etario,70+,NaN,34.0,3.32,14.0,20.0,41.18,58.82


,grupo,categoria,n_original,target_0_n_original,target_0_pct_original,target_1_n_original,target_1_pct_original,n_unicos,target_0_n_unicos,target_0_pct_unicos,target_1_n_unicos,target_1_pct_unicos
0,sex,0,312,86,27.56,226,72.44,96,24,25.00,72,75.00
1,sex,1,713,413,57.92,300,42.08,206,114,55.34,92,44.66
2,age_group,40-49,237,80,33.76,157,66.24,72,22,30.56,50,69.44
3,age_group,50-59,422,216,51.18,206,48.82,125,60,48.00,65,52.00
4,age_group,60-69,275,174,63.27,101,36.73,80,48,60.00,32,40.00
5,age_group,70+,34,14,41.18,20,58.82,10,4,40.00,6,60.00
6,age_group,<40,57,15,26.32,42,73.68,15,4,26.67,11,73.33


## 4. Intersección sex × edad

La tabla muestra el tamaño original y deduplicado de cada combinación. Para subgrupos con N deduplicado reducido, los porcentajes de target son sólo descriptivos y no prueban diferencias entre grupos.

In [5]:
subgroups = outputs['representatividad_subgrupos']
display(subgroups)
save_subgroup_heatmap(subgroups, 'Representación sex × rango etario (N deduplicado)', FIGURES_DIR / 'representacion_sex_x_edad.png')

,sex,age_group,n_original,porcentaje_original,n_unicos,porcentaje_unicos,target_0_n,target_1_n,target_0_pct,target_1_pct,target_0_n_original,target_1_n_original,observacion
0,0,<40,17,1.66,5,1.66,0,5,0.00,100.00,0,17,Subgrupo con N deduplicado reducido; los porce...
1,0,40-49,59,5.76,19,6.29,1,18,5.26,94.74,4,55,Tamaño disponible para descripción; no demuest...
2,0,50-59,109,10.63,34,11.26,10,24,29.41,70.59,35,74,Tamaño disponible para descripción; no demuest...
3,0,60-69,110,10.73,33,10.93,13,20,39.39,60.61,47,63,Tamaño disponible para descripción; no demuest...
4,0,70+,17,1.66,5,1.66,0,5,0.00,100.00,0,17,Subgrupo con N deduplicado reducido; los porce...
5,1,<40,40,3.90,10,3.31,4,6,40.00,60.00,15,25,Tamaño disponible para descripción; no demuest...
6,1,40-49,178,17.37,53,17.55,21,32,39.62,60.38,76,102,Tamaño disponible para descripción; no demuest...
7,1,50-59,313,30.54,91,30.13,50,41,54.95,45.05,181,132,Tamaño disponible para descripción; no demuest...
8,1,60-69,165,16.10,47,15.56,35,12,74.47,25.53,127,38,Tamaño disponible para descripción; no demuest...
9,1,70+,17,1.66,5,1.66,4,1,80.00,20.00,14,3,Subgrupo con N deduplicado reducido; los porce...


## 5. Principales hallazgos, limitaciones y mitigaciones

Cada registro distingue la evidencia observada de su interpretación, de los impactos potenciales y de la limitación de la evidencia. Ningún hallazgo afirma discriminación ni sesgo demostrado.

In [6]:
display(outputs['revision_variables_adicionales'])
display(outputs['hallazgos_eda'])
print('Tablas exportadas en:', TABLES_DIR)
print('Gráficos nuevos exportados en:', FIGURES_DIR)

,variable,categorias_observadas_original,categoria_mayoritaria_original,porcentaje_categoria_mayoritaria_original,maxima_diferencia_pp_original_vs_unicos,observacion
0,cp,4,0,48.49,1.14,Revisión descriptiva: requiere contraste con d...
1,fbs,2,0,85.07,0.03,Revisión descriptiva: requiere contraste con d...
2,restecg,3,1,50.05,0.19,Revisión descriptiva: requiere contraste con d...
3,exang,2,0,66.34,0.88,Revisión descriptiva: requiere contraste con d...
4,slope,3,1,47.02,0.93,Revisión descriptiva: requiere contraste con d...
5,ca,5,0,56.39,1.56,Revisión descriptiva: requiere contraste con d...
6,thal,4,2,53.07,1.56,Revisión descriptiva: requiere contraste con d...


,id,dimension,dataset_evaluado,evidencia,valor_observado,interpretacion,posible_sesgo,impacto_tecnico_potencial,impacto_etico_potencial,limitacion_de_la_evidencia,mitigacion_propuesta,requiere_revision
0,S01,Calidad de datos,Original,723 de 1025 filas son duplicados exactos al co...,70.54%,Existe una proporción alta de observaciones re...,Posible problema de calidad o muestreo; no se ...,El entrenamiento o análisis sin tratamiento po...,Podría afectar la confiabilidad y representati...,La coincidencia exacta no permite determinar s...,"Conservar el original, analizar la versión ded...",Sí
1,S02,Representación por sex,Original y deduplicado,sex=0 representa 30.44% del dataset original.,sex=0: 30.44%,La muestra presenta una distribución desigual ...,Posible fuente de sesgo de representación.,Un modelo futuro podría disponer de menor evid...,Requiere evaluar resultados por subgrupo antes...,No existe documentación local que permita asig...,"Mantener los códigos, verificar su definición ...",Sí
2,S03,Representación por edad,Original y deduplicado,El rango <40 concentra 5.56% del dataset origi...,<40: 5.56%,El rango etario posee pocas observaciones resp...,Posible fuente de representación desigual; no ...,Menor evidencia disponible para estimar patron...,Podría limitar la capacidad de generalización ...,La distribución de la muestra no permite infer...,"Documentar la limitación, contrastar con la po...",Sí
3,S04,Representación por edad,Original y deduplicado,El rango 70+ concentra 3.32% del dataset origi...,70+: 3.32%,El rango etario posee pocas observaciones resp...,Posible fuente de representación desigual; no ...,Menor evidencia disponible para estimar patron...,Podría limitar la capacidad de generalización ...,La distribución de la muestra no permite infer...,"Documentar la limitación, contrastar con la po...",Sí
4,S05,Intersección sex × edad,Deduplicado,"La combinación sex=0, <40 posee 5 observacione...",N único: 5,Es el subgrupo interseccional con menor eviden...,Posible limitación de representatividad inters...,Los porcentajes de target y futuras métricas e...,Se requiere cautela al generalizar conclusione...,El análisis no establece el tamaño o composici...,Reportar N junto a porcentajes y requerir vali...,Sí


Tablas exportadas en: C:\Users\cesar\OneDrive\Desktop\DuocUC\3erYear\GestionDeProyectoDeDatos\eva1\outputs\tables
Gráficos nuevos exportados en: C:\Users\cesar\OneDrive\Desktop\DuocUC\3erYear\GestionDeProyectoDeDatos\eva1\outputs\figures
